# Step 10 — CTRL-CNNVIT-V2-S42: repair the zero-output CNN

**Experiment type:** one-factor engineering control, NOT a proposed new method.

Our independently implemented SCV CNN–ViT (V0) yielded 55.8664% macro-F1 internal validation, and frozen diagnosis revealed a fully zero-output CNN final ReLU on 1,600 examples. The separately trained ViT-only control (V1) achieved 57.4409% on identical 35,005 training-derived validation targets.

**V2 tests only one architectural change**: `SCVIDS.cnn.layers[11]` final `ReLU()` is replaced by `LeakyReLU(negative_slope=0.1)`. All convolution weights/widths, ViT/patches, bidirectional attention, fusion/classifier, focal loss, AdamW, train-fitted preprocessing, 16×16 RGB serialization, and exact original train/validation split are unchanged. V2 trains from a fresh initialization and saves in its own namespace; V0 and V1 remain immutable.

**Safety gates:** Before fitting, exact source CSV and archived split/prediction SHA-256 hashes must match. Never open the designated official testing CSV. After each of 50 epochs, the notebook measures CNN token nonzero rate, between-example variance, and change in logits/predictions from zeroing CNN tokens on a **fixed class-aware 512-window subset** of original internal validation. Minimum health thresholds (predeclared, not tuned) are nonzero fraction 0.05 and mean feature std 1e-5. A failed gate remains a reported experimental outcome, never fabricated or silently discarded.

**Outputs:** all 50 epoch logs, resuming checkpoints (best, last, every 5 epochs), best prediction probabilities, 18 reproducible figure families × PNG/SVG/PDF = 54 figures, eight aggregate plot CSV tables, strict provenance and training package freeze. The full weights/probabilities archive goes to an unpublished draft GitHub Release, while aggregate figures/metrics go to the public repository. If postprocessing fails, rerun **only** the finalization cell, not the 50-epoch training.

**Restrictions:** Kaggle GPU & Internet on, attach dataset `dsshayan/unsw-nb15`, enable existing `pushRare` secret. This is *internal validation*, not test superiority. CSV adjacency has not been established chronological; window=1/permutation controls will follow.

In [ ]:
# Cell 1 — Validate Kaggle resources and restore original split/fit from pinned archive.
from pathlib import Path
import subprocess, os, sys
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-ctrl002")
BASE=Path("/kaggle/working/research_outputs")
RUN=BASE/"CTRL-CNNVIT-V2-S42"
WEIGHTS=BASE/"checkpoints_ctrl_cnnvit_v2_s42"
ARCHIVE=Path("/kaggle/working/SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar")
DATA=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
URL="https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"

assert DATA.is_file(),"Attach verified dsshayan/unsw-nb15 dataset before starting."
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",URL,str(REPO)],check=True)
print("Code commit:",subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())
import torch
assert torch.cuda.is_available(),"Enable Kaggle GPU accelerator"
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                "--format=csv,noheader"],check=True)

ENV=os.environ.copy()
ENV["PYTHONPATH"]=str(REPO/"src")
ENV["OPENBLAS_NUM_THREADS"]="1"
ENV["MKL_NUM_THREADS"]="1"
ENV["OMP_NUM_THREADS"]="1"
ENV["MPLBACKEND"]="Agg"

secret=UserSecretsClient().get_secret("pushRare")
SECURE_ENV=ENV.copy()
SECURE_ENV["GITHUB_TOKEN"]=secret
try:
    subprocess.run([sys.executable,str(REPO/"scripts"/"restore_scv_release.py"),
        "--base",str(BASE),"--archive",str(ARCHIVE)],
        cwd=str(REPO),env=SECURE_ENV,check=True)
finally:
    SECURE_ENV.pop("GITHUB_TOKEN",None)
    del secret
print("FROZEN REFERENCE SPLIT AND CHECKPOINT AVAILABLE.")

In [ ]:
# Cell 2 — The ONLY training cell: 50 epochs, best/last/five-epoch checkpoints.
CONFIG=REPO/"configs"/"ctrl002_cnnvit_leaky_seed42.json"
subprocess.run([sys.executable,str(REPO/"scripts"/"train_ctrl002_cnnvit.py"),
    "--repo",str(REPO),"--config",str(CONFIG),
    "--output",str(RUN),"--checkpoints",str(WEIGHTS)],
    cwd=str(REPO),env=ENV,check=True)
print("CTRL-V2 50-EPOCH TRAINING DONE; do not rerun for a plotting error.")

In [ ]:
# Cell 3 — NO TRAINING: frozen predictions -> metrics, numeric CSVs, 54 publication figures.
subprocess.run([sys.executable,str(REPO/"scripts"/"finalize_ctrl002_cnnvit.py"),
    "--run",str(RUN),"--checkpoints",str(WEIGHTS),
    "--reference",str(BASE/"SCV-IDS-REPRO-001")],
    cwd=str(REPO),env=ENV,check=True)
print((RUN/"run_report.md").read_text(encoding="utf-8"))

In [ ]:
# Cell 4 — Preserve private weights and then sync aggregate metrics/54 figures.
secret=UserSecretsClient().get_secret("pushRare")
SECURE_ENV=ENV.copy()
SECURE_ENV["GITHUB_TOKEN"]=secret
try:
    subprocess.run([sys.executable,str(REPO/"scripts"/"archive_ctrl002_cnnvit.py"),
        "--run",str(RUN),"--checkpoints",str(WEIGHTS),
        "--archive","/kaggle/working/CTRL-CNNVIT-V2-S42_FULL_ARTIFACTS.tar"],
        cwd=str(REPO),env=SECURE_ENV,check=True)
    subprocess.run([sys.executable,str(REPO/"scripts"/"sync_ctrl002_cnnvit.py"),
        "--run",str(RUN)],cwd=str(REPO),env=SECURE_ENV,check=True)
finally:
    SECURE_ENV.pop("GITHUB_TOKEN",None)
    del secret
print("V2 archived and source-data-backed images synchronized.")

### Send us both verification links

1. `VERIFIED CTRL-V2 SYNC COMMIT: https://github.com/...`
2. `CTRL-V2 PRIVATE RELEASE VERIFIED: https://github.com/...`

If epoch 50 finished but the finalizer or upload fails, **do not run all cells again**. Send the error, preserve the checkpoint folder, and only rerun Cell 3 or Cell 4 after a versioned fix. This study produces no official-test results and does not support a novel or high-impact-journal superiority claim by itself.